# Typist-ML: Pilot Results & Outcome Analysis

This notebook reproduces the complete primary and secondary outcome analysis for the Typist-ML randomized controlled trial (RCT) pilot, per Design Doc §§7, 15.1, §15.3 and Architectural Decisions:
- **D01 (Participant isolation):** Within-subject analysis only; no pooling across participants.
- **D05 (Beta credible intervals):** Equal-tailed 95% Jeffreys Beta intervals on set error rates.
- **D18 (Honesty & unblinded builder disclosure):** Explicit badging that the builder was unblinded.
- **D20 (Pre-registered analysis plan):** Strict fidelity to the analysis plan and seed reproducibility.

In [ ]:
import sys
from pathlib import Path

# Add services/api/src to sys.path for interactive execution
repo_root = Path.cwd().resolve()
if (repo_root / "services" / "api" / "src").exists():
    src_dir = repo_root / "services" / "api" / "src"
elif (repo_root.parent / "services" / "api" / "src").exists():
    src_dir = repo_root.parent / "services" / "api" / "src"
else:
    src_dir = Path("/Users/allen/typist-ml/services/api/src")

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sqlmodel import Session
from typist.config import get_settings
from typist.db import create_db_engine
from typist.services.analysis import (
    DEFAULT_BOOTSTRAP_RESAMPLES,
    DEFAULT_POSTERIOR_DRAWS,
    DEFAULT_SEED,
    overview_report,
    participant_results,
)
from typist.services.figures import (
    format_generation_table,
    format_limitations_paragraph,
    format_overhead_line,
    generate_all_figures,
    plot_set_error_rate_change,
)

## Section 1: Study Participants & Blinding Audit (D01, D18)

We inspect enrolled and completed participants from the study database, verifying the blinding status and completed session counts. Under D18, the system builder is explicitly flagged as unblinded.

In [ ]:
settings = get_settings()
engine = create_db_engine(settings.db_path)

with Session(engine) as session:
    overview = overview_report(session)

print(f"Participants found: {len(overview.participants)}")
for p in overview.participants:
    role = "Builder (unblinded)" if p.is_builder else "Peer (blinded)"
    print(f"- {p.alias} ({p.id}): status={p.status}, sessions={p.completed_sessions}, role={role}")

## Section 2: Data Quality & Pre-registered Exclusions (D20, D23)

Before evaluating primary outcomes, we audit session eligibility, abandoned/superseded session exclusions, dropped bigrams (<20 pooled retest attempts), and pause pairs (>2000 ms).

In [ ]:
completed_participants = [p for p in overview.participants if p.status == "done"]

if not completed_participants:
    print("No participants with status 'done' yet in database.")
else:
    with Session(engine) as session:
        for p in completed_participants:
            res = participant_results(session, p.id, seed=DEFAULT_SEED)
            print(f"Participant {res.alias}:")
            if res.exclusions:
                print(f"  Excluded sessions: {len(res.exclusions.excluded_sessions)}")
                print(f"  Dropped bigrams (<20 attempts): {len(res.exclusions.dropped_bigrams)}")
                print(f"  Pause pairs: {sum(res.exclusions.pause_counts.values())}")

## Section 3: Primary Outcome: Per-Set Error Rate Change & Beta Posterior Credible Intervals (D05, D20)

For each experimental set, we calculate the error rate change $d(s) = p(s, \text{retest}) - p(s, \text{baseline})$ and the targeting effect $E(s) = d(s) - d(\text{control})$ with 95% Jeffreys Beta credible intervals.

In [ ]:
if completed_participants:
    with Session(engine) as session:
        for p in completed_participants:
            res = participant_results(session, p.id, seed=DEFAULT_SEED)
            role_badge = "Builder (unblinded)" if res.is_builder else "Peer (blinded)"
            print(f"\nPrimary Outcomes for {res.alias} [{role_badge}]:")
            print(f"{'Set':4s} | {'Arm':10s} | {'Base Err':9s} | {'Retest Err':10s} | {'Change d(s)':12s} | {'Effect E(s)':12s}")
            print("-" * 70)
            for s in res.sets:
                b_str = f"{s.baseline_error_rate * 100:.1f}%" if s.baseline_error_rate is not None else "-"
                r_str = f"{s.retest_error_rate * 100:.1f}%" if s.retest_error_rate is not None else "-"
                d_str = f"{s.error_rate_change * 100:+.2f}%" if s.error_rate_change is not None else "-"
                e_str = f"{s.targeting_effect * 100:+.2f}%" if s.targeting_effect is not None else "control"
                print(f"{s.set_label:4s} | {s.arm:10s} | {b_str:9s} | {r_str:10s} | {d_str:12s} | {e_str:12s}")

## Section 4: Key Secondary Outcome: Median IKI & Non-Parametric Bootstrap 95% CIs (D20)

We evaluate Inter-Keystroke Interval (IKI) changes for correct adjacent transitions: $c(s) = \text{median\_iki}(s, \text{retest}) - \text{median\_iki}(s, \text{baseline})$ and targeting effect $F(s) = c(s) - c(\text{control})$ with 10,000 bootstrap resamples.

In [ ]:
if completed_participants:
    with Session(engine) as session:
        for p in completed_participants:
            res = participant_results(session, p.id, seed=DEFAULT_SEED)
            print(f"\nSecondary Outcomes (Median IKI) for {res.alias}:")
            print(f"{'Set':4s} | {'Arm':10s} | {'Base IKI':10s} | {'Retest IKI':11s} | {'Change c(s)':13s} | {'Effect F(s)':13s}")
            print("-" * 75)
            for s in res.sets:
                b_iki = f"{s.baseline_iki_median_ms:.1f} ms" if s.baseline_iki_median_ms is not None else "-"
                r_iki = f"{s.retest_iki_median_ms:.1f} ms" if s.retest_iki_median_ms is not None else "-"
                c_iki = f"{s.iki_change_ms:+.1f} ms" if s.iki_change_ms is not None else "-"
                f_iki = f"{s.iki_targeting_effect_ms:+.1f} ms" if s.iki_targeting_effect_ms is not None else "control"
                print(f"{s.set_label:4s} | {s.arm:10s} | {b_iki:10s} | {r_iki:11s} | {c_iki:13s} | {f_iki:13s}")

## Section 5: Fairness Gate & Generation Audit Table (§15.3, D16)

Audit of generator outputs across all arms: drill counts, mean target density (target $[0.10, 0.40]$), mean real-word ratio ($\ge 0.80$), and acceptance rate.

In [ ]:
gen_table = format_generation_table(overview)
print(gen_table)

## Section 6: Instrumentation Overhead & Throughput (§15.3)

Evaluation of handler runtime p95 (< 1.0 ms requirement) and dispatch lag p95, alongside Apple Silicon neural throughput for Arm.LLM.

In [ ]:
overhead_summary = format_overhead_line(overview)
print(overhead_summary)

## Section 7: README Artifact Regeneration (§15.3)

Regenerates the study figures and markdown snippets:
- `docs/figures/set_error_rate_change.png` (300 DPI high resolution)
- `docs/figures/set_error_rate_change.svg` (vector format)
- `docs/figures/readme_snippets.md` (summary table, overhead line, limitations paragraph)

In [ ]:
if completed_participants:
    with Session(engine) as session:
        artifacts = generate_all_figures(session)
        print(f"Saved primary chart: {artifacts.primary_png}")
        print(f"Saved vector SVG: {artifacts.primary_svg}")
        print(f"Saved snippets: {artifacts.readme_snippets_path}")
        print(f"Limitations snippet:\n{format_limitations_paragraph(session, overview)}")
else:
    print("Database has no completed participants yet; run `just dev` or complete pilot sessions to regenerate live figures.")